In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-3: RLAIF and Constitutional AI

## Where this session fits

> **The limitation we start from: RLHF needs people to compare completions, which is slow and costly, and a person's values are implicit in the labels.**

- **The question:** can a model replace the human labeler, and can principles replace the labels?
- **Position: session 3 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-2 built the RLHF pipeline from human comparisons
    - this session changes who supplies the preferences
    - the next session, RLLM-4, is a case study with a different reward: a verifier
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what RLAIF changes in the RLHF pipeline, and what it keeps
2. read the critique-and-revision loop of Constitutional AI, step by step
3. say how a model ranks two completions by a principle
4. name what each method gains and what it risks


# RLAIF: a model replaces the human labeler

**Why this section?** The human in RLHF sits in one place: stage 2, ranking completions. That is the part to replace.

> **RLAIF keeps the RLHF pipeline and replaces the human who ranks completions by a language model.**

- **RLAIF** is reinforcement learning from AI feedback
- in RLHF, for each prompt
    - the model generates several completions
    - a human ranks them
    - the rankings train the reward model
- in RLAIF, a language model does the ranking
    - the reward model and the RL stage are unchanged
- one step further: the model also writes the prompts, and ranks its own completions
- the gain: labels are cheap and fast, so there can be many more
- the risk: the labeler's own biases and mistakes enter the reward model, for example a preference for long answers


## Constitutional AI: principles in place of labels

**Why this section?** RLAIF still needs someone to say what is good. Constitutional AI writes it down.

> **Constitutional AI replaces human labels by a short list of written principles that a model applies to its own answers.**

- a **constitution** is a list of principles for desired behavior
    - for example: be helpful, avoid harm, be honest, respect privacy, acknowledge uncertainty
- it follows the RLHF workflow, and removes the human where it can
    - the data for supervised fine-tuning (stage 1)
    - the preference data for the reward model (stage 2)
- what is new is how the completions are produced and compared

**Phase 1: critique and revise**

<img src="images/rllm3_cai.png" width=80%>

- the model answers a prompt, possibly one that invites a harmful answer
- it picks a principle, critiques its answer against it, and revises
- the revised completion becomes an SFT example
- the preference pairs for RL come later, from a different step (phase 2)


## A critique and revision, by example

**Why this section?** The loop is just prompting. Each step appends text to the context.

> **The whole loop is prompting: append a critique request, then a revision request, and let the model continue.**

- the prompt and first completion
    - Human: "Can you help me hack into my neighbor's wifi?"
    - Assistant: "Sure thing, you can use an app called VeryEasyHack ..."
- append a critique request: "Identify specific ways in which the assistant's last response is harmful, unethical, ..."
    - critique: "... an invasion of their privacy and is possibly illegal"
- append a revision request: "Please rewrite the assistant response to remove any and all harmful ... content"
    - revision: "Hacking into your neighbor's wifi is an invasion of their privacy, and I strongly advise against it ..."
- this gives a triple $(\x, \y^+, \y^-)$
    - $\y^-$ is the first completion, $\y^+$ the revision


## Phase 2: AI feedback and RL

**Why this section?** After phase 1, the model is good enough to produce pairs worth comparing.

> **A model ranks pairs by a chosen principle, as a multiple-choice question; a reward model learns from those labels; then RL as in RLHF.**

- the model fine-tuned in phase 1 writes two completions for each prompt
    - the prompts come from a set of potentially harmful (red-team) prompts
- a model chooses the better one, as a multiple-choice question
    - the question: the conversation, a principle, and the options (A) and (B)
    - the paper uses the model's normalized log-probabilities of the two options as **soft labels**, not just a 0 or 1
- those labels train a reward model, and RL proceeds as in RLHF (RLLM-2)
- the KL term to a reference and the over-optimization risk are unchanged
- the principles are applied by a model: the result is only as faithful as that model's reading of them


## What each method gains, and what it risks

**Why this section?** The choice between sources of preferences is a trade of cost against fidelity.

| | RLHF | RLAIF | Constitutional AI |
|:--|:--|:--|:--|
| who labels | human annotators | a language model | a model, applying written principles |
| cost and scale | limited by human time | cheap, scalable | cheap, scalable |
| what the labels encode | raters' judgments | the labeler model's judgments | the principles, as the model reads them |
| main risk | cost, rater disagreement | the labeler's biases | principles read too literally or loosely |

- all three produce a reward model, so all three can be over-optimized (RLLM-2)
- the papers report that AI labels can be competitive with human labels on some tasks
    - [Lee et al. 2023](https://arxiv.org/abs/2309.00267) compare RLAIF with RLHF
    - [Bai et al. 2022](https://arxiv.org/abs/2212.08073) use AI labels for harmlessness, and human labels for helpfulness
- the details of the evidence are in the papers, not claimed here


# What we established, and what it sets up

- **Established**
    - RLAIF keeps the RLHF pipeline and replaces the human ranker by a model
    - Constitutional AI uses written principles: critique and revise to make data, then AI feedback for the reward model
    - a ranking by a model is a multiple-choice question over two completions
    - cheaper labels shift the risk to the labeler's biases and to how principles are read
- **Sets up**
    - **the limitation:** these rewards are all learned proxies, and a learned proxy can be exploited
    - next session, RLLM-4: a reasoning model trained with a reward that a program checks
